In [ ]:
##Instalación previa

!pip install evaluate bert-score
!pip install sacrebleu
!pip install unbabel-comet

!pip uninstall -y tensorflow tensorflow-estimator keras
!pip uninstall -y protobuf
!pip install protobuf==3.20.3
!pip install sentencepiece
!pip install transformers --upgrade


!pip uninstall -y transformers comet-ml pytorch-lightning
!pip install transformers==4.30.2
!pip install unbabel-comet==2.2.1
!pip install pytorch-lightning==2.0.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
Found existing installation: keras 3.13.2
Uninstalling keras-3.13.2:
  Successfully uninstalled keras-3.13.2
Found existing installation: protobuf 4.25.9
Uninstalling protobuf-4.25.9:
  Successfully uninstalled protobuf-4.25.9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-hub 0.26.0 requires keras>=3.13, which is not installed.
unbabel-comet 2.2.7 requires protobuf<5.0.0,>=4.24.4, but you have protobuf 3.20.3 which is incompatible.
google-cloud-datastore 2.25.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
grpc-google-iam-v1 0.14.4 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatibl

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
^C


In [18]:
!pip uninstall -y wandb

Found existing installation: wandb 0.27.2
Uninstalling wandb-0.27.2:
  Successfully uninstalled wandb-0.27.2


In [1]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/train.csv"

df = pd.read_csv(path)
df = df[["source", "target"]].dropna()

Mounted at /content/drive


In [2]:
import re

def limpiar_texto(texto):
    return re.sub(r"[^\w\s-]", "", texto)

***PROCESO DE PREPROCESAMIENTO***

In [3]:
#Silabificador

VOCALES = "aeiou"
CONSONANTES_COMPUESTAS = ["ch", "sh", "ts"]
SUFIJOS = ["klu", "ru", "lu", "chri", "xlu"]

def dividir_palabra(palabra):
    silabas = []
    actual = ""
    i = 0

    while i < len(palabra):
        if i + 1 < len(palabra):
            par = palabra[i:i+2]
            if par in CONSONANTES_COMPUESTAS:
                actual += par
                i += 2
                continue

        letra = palabra[i]
        actual += letra

        if letra in VOCALES:
            silabas.append(actual)
            actual = ""

        i += 1

    if actual:
        silabas.append(actual)

    return silabas


def silabificar_yine(texto):
    palabras = texto.lower().split()
    resultado = []

    for palabra in palabras:
        if len(palabra) <= 4:
            resultado.append(palabra)
            continue

        partes = palabra.split("-")

        for parte in partes:
            sufijo_encontrado = False

            for sufijo in SUFIJOS:
                if parte.endswith(sufijo) and len(parte) > len(sufijo):
                    raiz = parte[:-len(sufijo)]
                    resultado.extend(dividir_palabra(raiz))
                    resultado.append(sufijo)
                    sufijo_encontrado = True
                    break

            if not sufijo_encontrado:
                resultado.extend(dividir_palabra(parte))

    return resultado

In [5]:
df["source_clean"] = df["source"].apply(limpiar_texto)
df["target_clean"] = df["target"].apply(limpiar_texto)


In [12]:
# =========================
# EXTRAER SUBWORDS Y SÍLABAS
# =========================
from collections import Counter
subword_counter = Counter()
for texto in df["source_clean"]:
    palabras = texto.split()
    for palabra in palabras:
        silabas = silabificar_yine(palabra)
        subword_counter.update(silabas)

print("SUBWORDS MÁS FRECUENTES:")
print(subword_counter.most_common(50))

# Guardamos los nuevos tokens en una variable global
nuevos_tokens = [token for token, count in subword_counter.most_common(50)]



SUBWORDS MÁS FRECUENTES:
[('wa', 183), ('ka', 172), ('ta', 160), ('ne', 138), ('ga', 138), ('gi', 123), ('pa', 89), ('ru', 81), ('na', 72), ('chi', 62), ('nu', 56), ('tka', 51), ('ya', 48), ('ge', 43), ('tu', 39), ('pi', 38), ('nka', 37), ('su', 30), ('ko', 29), ('wane', 28), ('ra', 28), ('po', 26), ('ni', 25), ('kle', 25), ('lu', 23), ('no', 23), ('wu', 23), ('yo', 23), ('tlo', 22), ('chri', 22), ('te', 21), ('we', 21), ('mama', 19), ('wiwi', 19), ('tya', 19), ('pga', 19), ('ma', 18), ('papa', 17), ('klu', 16), ('lo', 15), ('gita', 15), ('tna', 15), ('je', 15), ('tje', 15), ('ri', 14), ('tma', 14), ('tni', 14), ('pe', 14), ('t', 13), ('nro', 13)]


In [7]:
from datasets import Dataset
df_model = df[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
dataset = Dataset.from_pandas(df_model)
print(dataset)


Dataset({
    features: ['source', 'target'],
    num_rows: 372
})


***TOKENIZACIÓN***

In [13]:
#Tokenización NLLB

from transformers import NllbTokenizer

tokenizer = NllbTokenizer.from_pretrained(
    "facebook/nllb-200-distilled-600M"
)

tokenizer.src_lang = "spa_Latn"

# Agregar los nuevos subwords de Yine al vocabulario si no existen ya
tokens_filtrados = [token for token in nuevos_tokens if token not in tokenizer.get_vocab()]
print("Nuevos tokens reales a agregar:", len(tokens_filtrados))
tokens_agregados = tokenizer.add_tokens(tokens_filtrados)
print("Tokens agregados con éxito:", tokens_agregados)



Nuevos tokens reales a agregar: 10
Tokens agregados con éxito: 10


In [14]:
##Función de tokenización para todo el dataset

def tokenizar_ejemplo(example):
    model_inputs = tokenizer(
        example["source"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [15]:
#Aplicación de función de tokenización

tokenized_dataset = dataset.map(tokenizar_ejemplo)
tokenized_dataset = tokenized_dataset.remove_columns(["source", "target"])
tokenized_dataset.set_format("torch")

print(tokenized_dataset)

Map:   0%|          | 0/372 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 372
})


***ENTRENAMIENTO DEL MODELO***

In [16]:
##Traer el modelo NLLB

from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/nllb-200-distilled-600M"
)

model.to("cuda")
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


M2M100ScaledWordEmbedding(256214, 1024, padding_idx=1)

In [19]:
#Preparar el trainning

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    learning_rate=3e-5,
    save_strategy="epoch",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [21]:
# Evitar que torchvision interfiera en la librería datasets
import sys
sys.modules.pop("torchvision", None)
sys.modules.pop("torchvision.io", None)

# Aplicación de función de tokenización
tokenized_dataset = dataset.map(tokenizar_ejemplo)
tokenized_dataset = tokenized_dataset.remove_columns(["source", "target"])
tokenized_dataset.set_format("torch")

print(tokenized_dataset)


Map:   0%|          | 0/372 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 372
})


In [22]:
##Entrenamiento del modelo

trainer.train()

Step,Training Loss
500,4.918600
1000,0.536000


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=1116, training_loss=2.4788285190486565, metrics={'train_runtime': 346.511, 'train_samples_per_second': 3.221, 'train_steps_per_second': 3.221, 'total_flos': 151155545997312.0, 'train_loss': 2.4788285190486565, 'epoch': 3.0})

In [23]:
#Guardar 1era version

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4")

('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4/sentencepiece.bpe.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4/added_tokens.json')

In [41]:
# ==============================================================================
# CARGA AUTOMÁTICA DEL MODELO ENTRENADO (EVITA ENTRENAR DESDE CERO)
# ==============================================================================
import os
from transformers import AutoModelForSeq2SeqLM

path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_v1"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando modelo entrenado v4 desde: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
    model.to("cuda")
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando modelo entrenado previo desde: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
    model.to("cuda")
else:
    print("[ADVERTENCIA] No se encontró ningún checkpoint en Google Drive.")
    print("[INFO] Se continuará usando el modelo base en memoria.")


[INFO] Cargando modelo entrenado v4 desde: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4


In [25]:
##Función para realizar las traducciones de val.csv y test.csv

def traducir(texto):
    inputs = tokenizer(texto, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("spa_Latn"),
        max_length=50,
        num_beams=4
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

***PROCESO DE VALIDACIÓN DEL MODELO CON EL DATASET VAL.CSV***

In [26]:
## Cargar val.csv

path_val = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/val.csv"

df_val = pd.read_csv(path_val)
df_val = df_val[["source", "target"]].dropna()

In [27]:
#Preprocesamiento de val
df_val["source_clean"] = df_val["source"].apply(limpiar_texto)
df_val["target_clean"] = df_val["target"].apply(limpiar_texto)


In [30]:
from datasets import Dataset

# Usamos df_val (validación) y no df (entrenamiento)
df_val_model = df_val[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
val_dataset = Dataset.from_pandas(df_val_model)
print(val_dataset)



Dataset({
    features: ['source', 'target'],
    num_rows: 46
})


In [31]:
##Generación de predicciones

preds = []
refs = []

for i in range(len(val_dataset)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = val_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [32]:
##Revisar algunas predicciones

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", val_dataset[i]["target"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")

INPUT: katu ganunrotanu
REAL: con quién va a casarse
PRED: vi a los conejos y vi a los conejos
-----
INPUT: ga wa wiwi tora nika
REAL: y mi hermanito come corbina
PRED: vamos a ver el río
-----
INPUT: gi piklu papa
REAL: mi papá no le tenía miedo
PRED: mi papá no es ciego
-----
INPUT: wuya gi patewatanutka
REAL: anda no tengas más vergüenza
PRED: yo quisiera recoger leña
-----
INPUT: gi ge ponro peta
REAL: no has visto al ciempiés
PRED: no veo a los conejos
-----


***MÉTRICAS PARA VALIDAR VAL.CSV***



In [33]:
##BLEU

import evaluate
bleu = evaluate.load("bleu")

bleu_result = bleu.compute(predictions=preds, references=refs)
print("BLEU:", bleu_result)

BLEU: {'bleu': 0.02772973562056651, 'precisions': [0.1118421052631579, 0.046511627906976744, 0.018867924528301886, 0.006024096385542169], 'brevity_penalty': 1.0, 'length_ratio': 1.3217391304347825, 'translation_length': 304, 'reference_length': 230}


In [34]:
##CHRF

chrf = evaluate.load("chrf")

chrf_result = chrf.compute(predictions=preds, references=refs)
print("ChrF:", chrf_result)

ChrF: {'score': 15.23077550518394, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [35]:
from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt20-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(val_dataset)):
    data.append({
        "src": df_val.iloc[i]["source"],
        "mt": preds[i],
        "ref": val_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)
print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages

COMET: -1.1043033070214416


In [38]:
# =========================
# MÉTRICA METEOR (VALIDACIÓN)
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR (Validación):", meteor_result)
except Exception as e:
    print("Error al calcular METEOR en validación:", e)


METEOR (Validación): {'meteor': 0.10966754686585038}


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [37]:
# =========================
# MÉTRICA BERTSCORE (VALIDACIÓN)
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Validación - Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Validación - Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Validación - Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore en validación:", e)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERTScore Precision (Validación - Promedio): 0.70958160965339
BERTScore Recall (Validación - Promedio): 0.7191602432209513
BERTScore F1 (Validación - Promedio): 0.7140831195789835


In [42]:
## REENTRENAR EL MODELO, PARA REVISION DE MEJORA
# ==============================================================================
# CAMBIOS PARA EL REENTRENAMIENTO INCREMENTAL:
# 1. learning_rate=3e-5 (Se reduce a 3e-5 en comparación con el valor inicial de 5e-5)
#    para realizar un ajuste fino más conservador y evitar el olvido catastrófico de los pesos
#    ya optimizados en la fase 1.
# 2. num_train_epochs = 3 (Se mantiene en 3 épocas para el refinamiento adicional).
# ==============================================================================

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    learning_rate=3e-5,
    save_strategy="epoch"
)


In [43]:
##Limpiamos el entorno

import torch
torch.cuda.empty_cache()

In [44]:
import torch
import gc

del model
del trainer

gc.collect()
torch.cuda.empty_cache()

In [45]:
from transformers import AutoModelForSeq2SeqLM
import os

# ==============================================================================
# REENTRENAMIENTO INCREMENTAL RESILIENTE:
# Busca los pesos de la Fase 1 en Google Drive (v4 o original) para continuar el aprendizaje.
# Si no encuentra ninguno, cae al modelo base de Hugging Face por seguridad.
# ==============================================================================
path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v1"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando pesos de Fase 1 (v4) para reentrenamiento: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando pesos de Fase 1 (original) para reentrenamiento: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
else:
    print("[ADVERTENCIA] No se encontró modelo previo. Cargando modelo base desde cero...")
    model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

model.to("cuda")
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


[INFO] Cargando pesos de Fase 1 (v4) para reentrenamiento: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4


M2M100ScaledWordEmbedding(256214, 1024, padding_idx=1)

In [46]:
##Creamos del trainer despues de limpiar memoria

from transformers import Trainer
from transformers import TrainingArguments

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [48]:
# Parche de compatibilidad: torchvision en Colab puede no tener VideoReader
try:
    import torchvision.io
    if not hasattr(torchvision.io, 'VideoReader'):
        class _DummyVideoReader:
            pass
        torchvision.io.VideoReader = _DummyVideoReader
except ImportError:
    pass

tokenized_dataset.set_format("torch")

In [49]:
#Realizamos el entrenamiento despues de los siguientes cambios de parametros

trainer.train()

Step,Training Loss
500,0.246600
1000,0.144400


TrainOutput(global_step=1116, training_loss=0.1928003765776166, metrics={'train_runtime': 352.6397, 'train_samples_per_second': 3.165, 'train_steps_per_second': 3.165, 'total_flos': 151155545997312.0, 'train_loss': 0.1928003765776166, 'epoch': 3.0})

In [50]:
##Guardar 2da version

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4_retrained")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4_retrained")

('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4_retrained/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4_retrained/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4_retrained/sentencepiece.bpe.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_NLLB_yine_v4_retrained/added_tokens.json')

In [51]:
# Carga y preparación del dataset de validación para evaluar el modelo después del reentrenamiento
print("Generando predicciones sobre el conjunto de validación con el modelo reentrenado...")

#Probar ejemplos

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", val_dataset[i]["target"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")


Generando predicciones sobre el conjunto de validación con el modelo reentrenado...
INPUT: katu ganunrotanu
REAL: con quién va a casarse
PRED: pero yo quisiera recoger ciempiés
-----
INPUT: ga wa wiwi tora nika
REAL: y mi hermanito come corbina
PRED: también veo a mi hermana
-----
INPUT: gi piklu papa
REAL: mi papá no le tenía miedo
PRED: mi papá no tiene miedo
-----
INPUT: wuya gi patewatanutka
REAL: anda no tengas más vergüenza
PRED: no has matado a un paujil
-----
INPUT: gi ge ponro peta
REAL: no has visto al ciempiés
PRED: no veo al ciempiés
-----


In [52]:
#Generar predicciones

preds = []
refs = []

for i in range(len(val_dataset)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = val_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [53]:
##Aplicamos nuevamente las métricas


##BLEU

import evaluate

bleu = evaluate.load("bleu")
bleu.compute(predictions=preds, references=refs)


{'bleu': 0.10724227441779882,
 'precisions': [0.27049180327868855,
  0.13131313131313133,
  0.06578947368421052,
  0.05660377358490566],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0608695652173914,
 'translation_length': 244,
 'reference_length': 230}

In [54]:
##CHRF

chrf = evaluate.load("chrf")

chrf_result = chrf.compute(predictions=preds, references=refs)

print("ChrF:", chrf_result)

ChrF: {'score': 25.6873654607563, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [55]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(val_dataset)):
    data.append({
        "src": df_val.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": val_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiment

COMET: 0.576319654998572


In [56]:
# =========================
# MÉTRICA METEOR (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR (Validación - Reentrenado):", meteor_result)
except Exception as e:
    print("Error al calcular METEOR en validación con reentrenamiento:", e)


METEOR (Validación - Reentrenado): {'meteor': 0.20376415900715994}


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [57]:
# =========================
# MÉTRICA BERTSCORE (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Validación - Reentrenado - Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Validación - Reentrenado - Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Validación - Reentrenado - Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore en validación con reentrenamiento:", e)


BERTScore Precision (Validación - Reentrenado - Promedio): 0.7538812251194663
BERTScore Recall (Validación - Reentrenado - Promedio): 0.7590513112752334
BERTScore F1 (Validación - Reentrenado - Promedio): 0.7562346277029618


Metricas para Test.csv

In [58]:
##PROBAMOS EL TEST.CSV

path_test = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/test.csv"

df_test = pd.read_csv(path_test)
df_test = df_test[["source", "target"]].dropna()

In [59]:
#Preprocesamiento de test
df_test["source_clean"] = df_test["source"].apply(limpiar_texto)
df_test["target_clean"] = df_test["target"].apply(limpiar_texto)


In [60]:
from datasets import Dataset

# ==============================================================================
# CREACIÓN DEL DATASET DE PRUEBA:
# Corregimos la NameError asegurando que se carga a partir de df_test (y no del df de entrenamiento).
# ==============================================================================
df_test_model = df_test[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
test_dataset = Dataset.from_pandas(df_test_model)
print("Dataset de prueba cargado con éxito:", test_dataset)


Dataset de prueba cargado con éxito: Dataset({
    features: ['source', 'target'],
    num_rows: 47
})


In [61]:
preds = []
refs = []

for i in range(len(test_dataset)):
    pred = traducir(df_test.iloc[i]["source_clean"])
    ref = test_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [62]:
##BLEU

bleu.compute(predictions=preds, references=refs)

{'bleu': 0.05442943499426501,
 'precisions': [0.24198250728862974,
  0.10135135135135136,
  0.03614457831325301,
  0.009900990099009901],
 'brevity_penalty': 1.0,
 'length_ratio': 1.1827586206896552,
 'translation_length': 343,
 'reference_length': 290}

In [63]:
##CHRF
chrf.compute(predictions=preds, references=refs)

{'score': 23.8076124446903, 'char_order': 6, 'word_order': 0, 'beta': 2}

In [64]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(test_dataset)):
    data.append({
        "src": df_test.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": test_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiment

COMET: 0.5579819279782315


In [65]:
# =========================
# MÉTRICA METEOR
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR:", meteor_result)
except Exception as e:
    print("Error al calcular METEOR:", e)


METEOR: {'meteor': 0.26852481262201006}


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [66]:
# =========================
# MÉTRICA BERTSCORE
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore:", e)


BERTScore Precision (Promedio): 0.7530491948127747
BERTScore Recall (Promedio): 0.7573973916946574
BERTScore F1 (Promedio): 0.7549051769236301
